In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.7 Checkpointing and resume

Real training runs get interrupted. A checkpoint saves the model **and** the
optimizer state **and** the step, so you can resume exactly where you stopped,
not restart. This produces the trained PocketLM the rest of the course uses.

In [ ]:
import tempfile
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, save_checkpoint, load_checkpoint)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(30):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
ckpt = os.path.join(tempfile.gettempdir(), "pocketlm_u5.pt")
save_checkpoint(ckpt, model, opt, step=30)
print("saved checkpoint at step 30")

Resume into a fresh model and optimizer: load restores the weights and reports
the step, so the next loop continues seamlessly.

In [ ]:
fresh = PocketLM(cfg)
fresh_opt = torch.optim.AdamW(fresh.parameters(), lr=1e-3)
resumed_step = load_checkpoint(ckpt, fresh, fresh_opt)
match = all(torch.equal(a, b) for a, b in zip(model.parameters(), fresh.parameters()))
print("resumed at step:", resumed_step, " weights match:", match)
assert resumed_step == 30
assert match